<a href="https://colab.research.google.com/github/rorisDS/workshop_ai_agents/blob/develop/notebooks_es/TuPrimerAgenteIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tu Primer Agente de IA

En este notebook vamos a construir y ejecutar un **Agente de IA extremadamente simple**, con un único objetivo: **entender cómo funciona un agente por dentro**.

La idea no es crear un sistema inteligente complejo ni “útil” en términos prácticos, sino **aislar los elementos mínimos** que hacen que un LLM deje de ser un simple modelo de conversación y pase a comportarse como un **agente**:

* Un **modelo de lenguaje** que razona sobre una petición.
* Un conjunto de **herramientas (tools)** que puede utilizar para actuar.
* Un **prompt de sistema** que define su rol y sus restricciones.
* Un **bucle de decisión** en el que el modelo decide cuándo y cómo usar esas herramientas.

Para ello, implementaremos un agente capaz de **resolver operaciones matemáticas básicas** utilizando herramientas explícitas (suma, multiplicación, cuadrado), en lugar de calcular directamente el resultado.

Este ejemplo nos permitirá observar:

- Cómo el agente interpreta una petición del usuario.
- Cómo decide **qué herramientas invocar** para resolverla.
- Cómo **encadena múltiples llamadas a tools** para llegar a una respuesta final.
- Qué información intercambia el agente con cada herramienta durante la ejecución.

A partir de este punto, el mismo patrón se puede extender a agentes mucho más complejos: acceso a bases de datos, APIs externas, sistemas RAG, planificación multi-paso o incluso interacción entre múltiples agentes.

## Instalación de librerías

In [ ]:
!pip install langchain==1.2.7
!pip install langchain-core==1.2.7
!pip install langchain-openai==1.1.7  # Para usar modelos de OpenAI
!pip install langchain-google-genai==4.2.0  # Para usar modelos de Google (Gemini)
!pip install ddgs==9.10.0
!pip install langchain-community==0.4.1

In [1]:
# Codigo auxiliar desarrollado para facilitar la visualizacion de los mensajes en el agente
import os

project_path = "/content/workshop_ai_agents"

if os.path.exists(project_path) == False:
  !git clone https://github.com/rorisDS/workshop_ai_agents

import sys
sys.path.append(project_path)

from utils.agent_message_pretty_debug import PrettyDebug

## LLM

In [11]:
# Use Google Colab Secrets
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    pass

* Conectando a un modelo de OpenAI

   - Crear API Key: https://platform.openai.com/docs/quickstart
   - Seleccionar un modelo: https://platform.openai.com/docs/models

In [12]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o",  # Seleccionamos un modelo por su keyword
    temperature=0,
    max_tokens=None,
    # other params...
    callbacks=[PrettyDebug()]   # <--- Verbose
    )

* Conectando a un modelo de Google
   - Crear API Key: https://ai.google.dev/gemini-api/docs/api-key
   - Seleccionar un modelo: https://ai.google.dev/gemini-api/docs/models

In [13]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.0,
    max_tokens=None,
    # other params...
    callbacks=[PrettyDebug()]   # <--- Verbose
)

## Tools

In [42]:
from langchain_core.tools import tool

# setup the tools
@tool
def suma(a: int, b: int) -> int:
    """Suma dos numeros."""
    return a + b


@tool
def multiplicacion(a: int, b: int) -> int:
    """Multiplica 2 numeros."""
    return a * b


@tool
def cuadrado(a: int) -> int:
    """Calcula el cuadrado de un numero."""
    return a * a

## Agente

In [43]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[suma, multiplicacion, cuadrado],
    system_prompt="""Eres un asistente de matematicas.
Utiliza las tools para responder las preguntas. Si no tienes una tool para responder una pregunta, deja constancia.
Devuelve solo el resultado de la operacion."""
)

In [44]:
from langchain.messages import HumanMessage

result = agent.invoke(
    {
        "messages": [
            HumanMessage("Cual es el resultado ((3+3)-2^2)*5?")
        ]
    }
)


▶ LLM START

Message:
System: Eres un asistente de matematicas.
Utiliza las tools para responder las preguntas. Si no tienes una tool para responder una pregunta, deja constancia. 

Devuelve solo la respuesta. Por ejemplo:
Human: Cuanto es 1 + 1?
AI: 2
Human: Cual es el resultado ((3+3)-2^2)*5?



RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [31]:
print(f"Respuesta: {result.content[0]['text']}")

{'messages': [HumanMessage(content='Cual es el resultado ((3+3)-2^2)*5?', additional_kwargs={}, response_metadata={}, id='4d44863d-c112-41c0-9b63-c442d33f15bf'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'cuadrado', 'arguments': '{"a": 2}'}, '__gemini_function_call_thought_signatures__': {'a61450d9-69d1-478f-81fe-c9f09f846bb5': 'EtUUCtIUAXLI2nz278CT/DajBam3qGMnmK1eVBEJbWrtnXLwo9el6MsHgTMTbl0etTvvdPNBtoRi6aabtWBmaJhQSvvlj27bC3Wn73WlX9hzqAcKqg9drCeLKcaHkp8EOImjN5jO/jfErOuSm9MWE+6GQohf6BuGoAxL1SsF7SuiwPdDGUpBdAXIWGLFdkcnF+w0eHtOyiQzodsFz+E2as0Lx9EzvPKErhka04xBYTMOrrR9qw/6zrUO7RWQ3i7I5wcpmweA99XPZkBGtoV4KmXnyN2ePIlK/JJn5DGqVTd5dwiftcTgEpYicPk6jEKBf4NkeF7JDOus4iPUQs1yh+0jg0J78LDkNqoeOPBrv/FMP2gn8gPdMSl6nC1oLekmooc+pIlyXguU/iRyjw8SIsRgf7xIWYbURH/+CwHglgR5h7FKSP5EF/O3JhWUbxCuabXJGnBbnxWCj3TM1x2IXllTzfHUmTYVIPZfT/rdjpJ6ihVlXGmy2KlHPbQgwPwdc2RGejXlRcWX/s/02zwLB6EzxygDl56iBI3o+MIxMBD5qNKjIetORI7uT4PxDyRJjXt4mO5teYFGSwxMUfnQyMAE2jkCmBKwzuO4N45Tnx9JQU2HaofHbMo09Y9CIqteN4